# Step 6.c — Climate and soil suitability under the SSP scenarios of EC-Earth3, global (EC-Earth3 grid)

The final rule of 5.h / 6.b applied to the training reference (`cu.REF_LABEL`, 1981–2014) and to the CMIP6 SSPs of
EC-Earth3 r1i1p1f1 (SSP1-2.6, 2-4.5, 3-7.0, 5-8.5; 2041–2070 and 2071–2100; delta BIO variables of 1.3.a against
EC-Earth3 historical 1981–2014, run with `CROP_SCEN=ssp`; SVM of 3.g, the same models as for the AMOC runs; climate
probability of 4.g): **suitable = climate suitable (`suit_bin` of 4.g: probability ≥ `cu.CLIMATE_P` at equal priors and
growing-period temperature ≥ the FAO ECOCROP TMIN) AND soil rating ≥ `cu.SOIL_MIN`**, per crop and GAEZ input level
(low, high). The soil is the same in every scenario: a change comes from the climate only.

Every run is compared with the reference: suitable land on all land and on the cells assessable in both (novel in
neither), land gained and lost, the share of today's crop area (MIRCA2000) that stays suitable, by region (as 6.b).
Novel climate (4.g: MESS < 0, NT2 > 1 or no support vector within the reach of the kernel): not suitable, reported apart.

Outputs (with `CROP_SCEN=ssp` under `scen_ssp/`) in `<out_root>/.../6_climate_soil_ssp/<route>/<variant>_<radiation>/`:
`climate_soil_scenarios_<crop>.nc` (`suitable_<level>`, `novel`, dimension `scenario`), `areas_scenarios_<set>.csv`,
`ssp_change_<set>.csv`, `ssp_change_regions_<set>.csv`, and the maps of section 3 in `<crop>/`.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the set of 4.g and 5.h
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # as 4.g and 5.h
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
assert cu.CROP_SCEN == "ssp", "6.c is the SSP application: run with CROP_SCEN=ssp (crop_utils), after 1.3.a and 4.g"
RUNS = cu.runs()                   # <ssp>_<first year>-<last year> (crop_utils.SCENARIO_SETS)
SSPS = sorted({r.split("_")[0] for r in RUNS})
WINDOWS = sorted({r.split("_")[1] for r in RUNS})
LEVELS = ["low", "high"]           # GAEZ input level of the soil rating (2.g)

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
OUT = cu.out_dir(CFG, "6_climate_soil_ssp", ROUTE, SET)
SCEN = [cu.REF_LABEL, *RUNS]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
LAND_MHA = LAND_HA * LAND / 1e6   # land area of the cells (Mha)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_{cu.GRID_TAG}.nc").load() for lv in LEVELS}
print(f"set {SET} | rule: climate p >= {cu.CLIMATE_P} and >= ECOCROP TMIN, soil >= {cu.SOIL_MIN} | climate {CLIM_DIR} | outputs {OUT}")

## 1. Suitability per scenario, and every SSP run against the reference

In [ ]:
mha = lambda a: float(np.nansum(LAND_MHA * a))
pct = lambda a, b: 100 * (a - b) / b if b else np.nan
area_rows, effect_rows, region_rows, MAPS = [], [], [], {}
for crop in CROPS:
    with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
        missing = [s for s in SCEN if s not in ds.scenario.values]
        assert not missing, f"{crop}: scenarios {missing} not in 4.g"
        clim_ok = ds["suit_bin"].sel(scenario=SCEN).values == 1               # (scenario, lat, lon): p >= 0.5, >= TMIN, not novel
        novel = ds["novel"].sel(scenario=SCEN).values == 1
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    dv, suit = {}, {}
    for lv in LEVELS:
        ssr = np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0)      # 0-100, no soil rating -> 0 (5.h)
        suit[lv] = LAND & cu.suitable(clim_ok, ssr)                           # (scenario, lat, lon)
        dv[f"suitable_{lv}"] = (("scenario", "lat", "lon"), np.where(LAND, suit[lv], np.nan).astype("float32"),
                                {"long_name": f"final suitability, {lv} input: climate suitable (p >= {cu.CLIMATE_P}, >= ECOCROP TMIN) and soil >= {cu.SOIL_MIN}"})
    dv["novel"] = (("scenario", "lat", "lon"), novel.astype("int8"), {"long_name": "novel climate (4.g)"})
    xr.Dataset(dv, coords={"scenario": SCEN, "lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "set": SET, "route": ROUTE, "climate_p": cu.CLIMATE_P, "soil_min": cu.SOIL_MIN}
               ).to_netcdf(OUT / f"climate_soil_scenarios_{crop}.nc")
    MAPS[crop] = {"suit": suit, "novel": novel, "clim": clim_ok}

    for k, s in enumerate(SCEN):
        r = {"crop": crop, "scenario": s, "novel (Mha)": mha(novel[k]),
             "MIRCA area of the crop in novel cells (%)": 100 * crop_ha[novel[k]].sum() / crop_ha.sum()}
        for lv in LEVELS:
            r[f"suitable, {lv} (Mha)"] = mha(suit[lv][k])
            r[f"today's crop area suitable, {lv} (%)"] = 100 * crop_ha[suit[lv][k]].sum() / crop_ha.sum()
        area_rows.append(r)

    e = SCEN.index(cu.REF_LABEL)
    for run in RUNS:
        k = SCEN.index(run)
        both = LAND & ~(novel[k] | novel[e])                                   # assessable in both
        for lv in LEVELS:
            sk, s0 = suit[lv][k], suit[lv][e]
            r = {"crop": crop, "run": run, "level": lv, "novel in either (Mha)": mha(LAND & ~both),
                 "suitable change, all land (%)": pct(mha(sk), mha(s0)),
                 "suitable change, assessable in both (%)": pct(mha(sk & both), mha(s0 & both)),
                 "gained (Mha)": mha(sk & ~s0 & both), "lost (Mha)": mha(s0 & ~sk & both),
                 "today's crop area suitable: change (points)": 100 * (crop_ha[sk].sum() - crop_ha[s0].sum()) / crop_ha.sum()}
            effect_rows.append(r)
            for reg, rm in REGIONS.items():
                a0, ak = mha(s0 & both & rm), mha(sk & both & rm)
                region_rows.append({"crop": crop, "run": run, "level": lv, "region": reg,
                                    "suitable, reference (Mha)": a0, "suitable change (Mha)": ak - a0,
                                    "suitable change (%)": pct(ak, a0) if a0 > 0.5 else np.nan})
    print(f"{crop}: done", flush=True)

## 2. Results

(1) Suitable land and share of today's crop area suitable, per scenario. (2) Every SSP run against the reference:
suitable land on all land and on the cells assessable in both, land gained and lost, change of the share of today's
crop area suitable. (3) The change by region (cells assessable in both).

In [ ]:
areas = pd.DataFrame(area_rows).set_index(["crop", "scenario"])
effect = pd.DataFrame(effect_rows).set_index(["crop", "run", "level"])
regions = pd.DataFrame(region_rows).set_index(["crop", "run", "level", "region"])
areas.to_csv(OUT / f"areas_scenarios_{SET}.csv"); effect.to_csv(OUT / f"ssp_change_{SET}.csv")
regions.to_csv(OUT / f"ssp_change_regions_{SET}.csv")
with pd.option_context("display.max_rows", 400, "display.max_columns", 40, "display.width", 250):
    print("(1) suitable land per scenario (Mha) and share of today's crop area suitable (%)")
    display(areas.round(2))
    for lv in LEVELS:
        print(f"(2) suitable land change from {cu.REF_LABEL} (%), cells assessable in both, {lv} input; rows: crop, columns: run")
        display(effect.xs(lv, level="level")["suitable change, assessable in both (%)"].unstack("run")[RUNS].round(1))
        print(f"(2b) today's crop area suitable: change from {cu.REF_LABEL} (points), {lv} input")
        display(effect.xs(lv, level="level")["today's crop area suitable: change (points)"].unstack("run")[RUNS].round(1))
        print(f"(3) suitable land change (%) by region, cells assessable in both, {lv} input")
        display(regions.xs(lv, level="level").reset_index().pivot_table(index=["crop", "run"], columns="region", sort=False,
                values="suitable change (%)")[list(REGIONS)].round(1))

## 3. Final binary maps: the SSP runs against the reference

Per crop and input level, in `<outputs>/<crop>/`:
1. `<crop>_<level>_1_<reference>.png`: the reference 1981–2014;
2. `<crop>_<level>_2_ssp_vs_<reference>.png`: land lost and gained from the reference, one row per period
   (2041–2070, 2071–2100), one column per SSP;
3. `<crop>_<level>_3_ssp_losses_on_today_cropland.png`: on the land where the crop is grown today (MIRCA2000, presences
   of 3.g), the cells still suitable, lost (suitable on the reference, not in the run), already not suitable on the
   reference ("not assessable: model miss") and with a climate never seen in training in the run; same layout.

Colours and the definition of "climate never seen in training" as in 6.b.

In [ ]:
MAP_LEVELS = ["low", "high"]       # GAEZ input level(s) of the maps
from matplotlib.patches import Patch
NA = "climate never seen in training"   # novel climate of 4.g
CAPTION = (f"Training: {cu.REF_LABEL} 1981-2014 climates of today's MIRCA2000 cropland, described over the growing season of the crop "
           "(GGCMI calendar). \"Climate never seen in training\" = outside them (MESS, NT2, kernel support): the model of "
           "this crop cannot assess the suitability there.")
BIN = (["#e6e6e6", "#1a9641", "#7f7f7f"], ["not suitable", "suitable", NA])
CHG = ["#e6e6e6", "#a6d96a", "#d7191c", "#1a9641", "#7f7f7f", "#9e9ac8"]   # + novel in the reference | novel only in the run
chg_labels = lambda ref, run: ["not suitable in both", "suitable in both", "lost", "gained",
                               f"{ref}: climate never seen in training", f"only {run}: climate never seen in training"]
MIR = (["#e6e6e6", "#1a9641", "#d7191c", "#a6d96a", "#7f7f7f"],
       ["neither", "suitable and cultivated", "cultivated, not suitable", "suitable, not cultivated", NA])
SOIL_MODERATE = 25                 # GAEZ class "moderate" or better, for the display of map 0 only
MIR3 = (["#e6e6e6", "#1a9641", "#d7191c", "#fdae61", "#a6d96a", "#fee08b", "#7f7f7f"],
        ["neither", "suitable and cultivated", "cultivated, not suitable: climate",
         "cultivated, not suitable: soil only (climate suitable)", f"suitable, not cultivated, soil >= {SOIL_MODERATE}",
         f"suitable, not cultivated, marginal soil ({cu.SOIL_MIN}-{SOIL_MODERATE})", NA])
SOIL_EDGES = [0, 10, 25, 40, 55, 70, 85, 100.01]   # GAEZ classes: not / very marginal, marginal, moderate, medium, good, high, very high
SOIL_COLS = ["#8c510a", "#d8b365", "#f6e8c3", "#c7eae5", "#80cdc1", "#35978f", "#01665e"]
TODAY = (["#e6e6e6", "#1a9641"], ["not cultivated today", "cultivated today (MIRCA2000)"])
LOSS = (["#e6e6e6", "#a6d96a", "#d7191c", "#bdbdbd", "#9e9ac8"],   # model miss in grey: context, not a result
        ["not cultivated today", "still suitable", f"lost (suitable on {cu.REF_LABEL})", f"not assessable: cultivated but not suitable on {cu.REF_LABEL} (model miss)"])
# the effect of the collapse (blue, red) in strong colours, the context (suitable in both, model miss, lost in both, novel) pale
AMOC_TODAY = (["#e6e6e6", "#c7e9c0", "#2166ac", "#d7191c", "#bdbdbd", "#d8b365", "#cbc9e2"],
              ["not cultivated today", "suitable with and without the collapse", "suitable only with the collapse (hosing)",
               "suitable only without the collapse (no hosing)", f"not assessable: cultivated but not suitable on {cu.REF_LABEL} (model miss)",
               f"lost with and without the collapse (suitable on {cu.REF_LABEL})", "climate never seen in training in either"])
NICE = {"Maize_rf": "maize", "Potato_rf": "potato", "Soybean_rf": "soybean", "Rice_rf": "rice",
        "Wheat_rf": "wheat (winter + spring)", "Wheat-winter_rf": "winter wheat", "Wheat-spring_rf": "spring wheat"}


def panel(ax, classes, colours, labels, title, outside=False):
    ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, classes, np.nan), vmin=-0.5, vmax=len(colours) - 0.5,
                  cmap=ListedColormap(colours), shading="nearest")
    ax.set_ylim(-60, 85); ax.set_title(title, fontsize=10)
    hs = [Patch(color=c, label=l) for c, l in zip(colours, labels)]
    if outside == "right":                        # right of the map
        ax.legend(handles=hs, loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9, framealpha=0.9)
    elif outside:                                 # in the empty space left of the map
        ax.legend(handles=hs, loc="center right", bbox_to_anchor=(-0.06, 0.5), fontsize=10, framealpha=0.9)
    else:
        ax.legend(handles=hs, loc="lower left", fontsize=7, framealpha=0.9)


def binary(k, S, N):
    return np.where(N[k], 2.0, S[k].astype(float))


def change(ref, new, S, N):
    # 0-3: not suitable / suitable in both, lost, gained; 4: novel in the reference; 5: novel only in the new climate
    a, b, nov = S[ref], S[new], N[ref] | N[new]
    c = np.where(a & b, 1.0, np.where(a & ~b, 2.0, np.where(~a & b, 3.0, 0.0)))
    c = np.where(N[ref], 4.0, np.where(N[new], 5.0, c))
    g, l = mha(~nov & ~a & b), mha(~nov & a & ~b)
    return c, g, l


def save(fig, path):
    import textwrap
    fig.text(0.5, -0.01, textwrap.fill(CAPTION, int(fig.get_figwidth() * 13)), ha="center", va="top", fontsize=9, style="italic")
    fig.savefig(path, dpi=110, bbox_inches="tight"); plt.close(fig)


written = []
e = SCEN.index(cu.REF_LABEL)
for crop in CROPS:
    D = OUT / crop; D.mkdir(exist_ok=True)
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres = LAND & (crop_ha > CELL_AREA_THR * np.nan_to_num(cu.land_area_ha(CFG)))   # presences of 3.g
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    for lv in MAP_LEVELS:
        S, N = MAPS[crop]["suit"][lv], MAPS[crop]["novel"]
        head = f"{NICE.get(crop, crop)}, {lv} input"
        fig, ax = plt.subplots(figsize=(11, 5.2))
        panel(ax, binary(e, S, N), *BIN, f"{head} (climate >= {cu.CLIMATE_P} and >= TMIN, soil >= {cu.SOIL_MIN}): "
              f"{cu.REF_LABEL} 1981-2014 | suitable {mha(S[e]):.0f} Mha")
        f = D / f"{crop}_{lv}_1_{cu.REF_LABEL}.png"; save(fig, f); written.append(f)
        fig, axes = plt.subplots(len(WINDOWS), len(SSPS), figsize=(10 * len(SSPS), 5.2 * len(WINDOWS)), squeeze=False)
        for i, w in enumerate(WINDOWS):
            for j, x in enumerate(SSPS):
                k = SCEN.index(f"{x}_{w}")
                c, g, l = change(e, k, S, N)
                panel(axes[i, j], c, CHG, chg_labels(cu.REF_LABEL, f"{x} {w}"),
                      f"{head}: {x} {w} vs {cu.REF_LABEL}: gained {g:.0f} Mha, lost {l:.0f} Mha")
        f = D / f"{crop}_{lv}_2_ssp_vs_{cu.REF_LABEL}.png"; save(fig, f); written.append(f)
        fig, axes = plt.subplots(len(WINDOWS), len(SSPS), figsize=(10 * len(SSPS), 5.2 * len(WINDOWS)), squeeze=False)
        for i, w in enumerate(WINDOWS):
            for j, x in enumerate(SSPS):
                k = SCEN.index(f"{x}_{w}")
                nov_k = pres & N[k]
                miss = pres & ~S[e] & ~nov_k              # the model misses them on the reference: nothing to say
                kept, lost = pres & S[e] & S[k], pres & S[e] & ~S[k] & ~nov_k
                c = np.where(nov_k, 4.0, np.where(miss, 3.0, np.where(kept, 1.0, np.where(lost, 2.0, 0.0))))
                panel(axes[i, j], c, LOSS[0], LOSS[1] + [f"climate never seen in training ({x} {w})"],
                      f"{head}, {x} {w} on today's cropland: lost {share(lost):.1f} % of the crop area, "
                      f"still suitable {share(kept):.1f} %, not assessable {share(miss):.1f} %")
        f = D / f"{crop}_{lv}_3_ssp_losses_on_today_cropland.png"; save(fig, f); written.append(f)
print(f"{len(written)} maps written in {OUT}/<crop>/")